# Notebook 09: Evaluation & Explainability

**Goal:** Deeply evaluate our LightGBM ranking model and make it interpretable using SHAP (SHapley Additive exPlanations).

**What We'll Build:**
- Comprehensive evaluation metrics
- SHAP explanations for individual recommendations
- Feature contribution analysis
- Model behavior insights
- Error analysis
- Recommendation explanations

**Evaluation Metrics:**
- NDCG@K (Normalized Discounted Cumulative Gain)
- Precision@K, Recall@K
- Diversity & Coverage
- Quality metrics (avg score, genre alignment)

**Explainability:**
- SHAP values for feature importance
- Per-recommendation explanations
- Feature interactions
- Visualization of model decisions

**Output:** Interpretable, trustworthy recommendation system

---

## 1. Setup and Load Model`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import lightgbm as lgb
import faiss
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ndcg_score
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

DATA_DIR = Path('data')
PROCESSED_DIR = DATA_DIR / 'processed'

print("NOTEBOOK 09: EVALUATION & EXPLAINABILITY")
print("="*70)

# Load data
df = pd.read_parquet(PROCESSED_DIR / 'anime_features.parquet')

# Load model
model = lgb.Booster(model_file=str(PROCESSED_DIR / 'lgbm_ranker.txt'))

# Load FAISS index
index = faiss.read_index(str(PROCESSED_DIR / 'faiss_index_ivf.bin'))

# Load embeddings
embeddings_combined = np.load(PROCESSED_DIR / 'embeddings_combined.npy')
graph_features = pd.read_parquet(PROCESSED_DIR / 'graph_features.parquet')

print("\nData loaded:")
print(f"  Anime: {len(df):,}")
print(f"  Model: LightGBM Ranker")
print(f"  FAISS index: {index.ntotal:,} vectors")

print("\nInstalling SHAP for explainability...")

try:
    import shap
    print(f"✓ SHAP version: {shap.__version__}")
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '--break-system-packages'])
    import shap
    print("✓ SHAP installed")

print("\n✓ Ready for evaluation!")

NOTEBOOK 09: EVALUATION & EXPLAINABILITY

Data loaded:
  Anime: 19,931
  Model: LightGBM Ranker
  FAISS index: 19,931 vectors

Installing SHAP for explainability...
✓ SHAP version: 0.49.1

✓ Ready for evaluation!


## 2. Comprehensive Evaluation Metrics

Evaluate the LTR model on multiple dimensions: ranking quality, diversity, and content relevance.


In [2]:
from sklearn.metrics.pairwise import cosine_similarity

print("COMPREHENSIVE MODEL EVALUATION")
print("="*70)

# Load feature extraction function from Notebook 08
def create_ranking_features(query_idx, candidate_idx):
    """Create features for ranking query-candidate pair"""
    query_emb = embeddings_combined[query_idx].reshape(1, -1)
    cand_emb = embeddings_combined[candidate_idx].reshape(1, -1)
    cosine_sim = cosine_similarity(query_emb, cand_emb)[0][0]
    
    cand_graph = graph_features.loc[candidate_idx].values
    cand_score = df.loc[candidate_idx, 'Score'] if pd.notna(df.loc[candidate_idx, 'Score']) else 6.5
    cand_members = df.loc[candidate_idx, 'log_members']
    cand_favorites = df.loc[candidate_idx, 'log_favorites']
    cand_popularity = df.loc[candidate_idx, 'members_percentile']
    
    query_genres = set(df.loc[query_idx, 'genres_list'])
    cand_genres = set(df.loc[candidate_idx, 'genres_list'])
    genre_overlap = len(query_genres & cand_genres)
    genre_jaccard = len(query_genres & cand_genres) / len(query_genres | cand_genres) if len(query_genres | cand_genres) > 0 else 0
    
    query_studios = set(df.loc[query_idx, 'studios_list'])
    cand_studios = set(df.loc[candidate_idx, 'studios_list'])
    studio_overlap = len(query_studios & cand_studios)
    
    has_score = 1 if pd.notna(df.loc[candidate_idx, 'Score']) else 0
    is_highly_rated = df.loc[candidate_idx, 'is_highly_rated']
    
    features = [
        cosine_sim, *cand_graph, cand_score, cand_members, cand_favorites,
        cand_popularity, genre_overlap, genre_jaccard, studio_overlap,
        has_score, is_highly_rated
    ]
    
    return np.array(features)


def get_ltr_recommendations(query_idx, k=10):
    """Get LTR model recommendations"""
    query_emb = embeddings_combined[query_idx].reshape(1, -1)
    n_candidates = k * 5
    distances, indices = index.search(query_emb, n_candidates + 1)
    
    mask = indices[0] != query_idx
    candidates = indices[0][mask][:n_candidates]
    
    candidate_features = np.array([
        create_ranking_features(query_idx, cand_idx)
        for cand_idx in candidates
    ])
    
    scores = model.predict(candidate_features)
    top_k_idx = np.argsort(scores)[::-1][:k]
    
    return candidates[top_k_idx], scores[top_k_idx]


# Evaluation metrics
def evaluate_recommendations(test_queries, k=10):
    """Compute comprehensive metrics"""
    results = {
        'avg_scores': [],
        'genre_overlaps': [],
        'diversities': [],
        'coverage': set()
    }
    
    for query_idx in test_queries:
        recs, _ = get_ltr_recommendations(query_idx, k=k)
        
        # Quality metrics
        rec_scores = [df.loc[r, 'Score'] if pd.notna(df.loc[r, 'Score']) else 6.5 for r in recs]
        results['avg_scores'].append(np.mean(rec_scores))
        
        # Genre overlap
        query_genres = set(df.loc[query_idx, 'genres_list'])
        overlaps = [len(set(df.loc[r, 'genres_list']) & query_genres) for r in recs]
        results['genre_overlaps'].append(np.mean(overlaps))
        
        # Diversity
        all_genres = set()
        for r in recs:
            all_genres.update(df.loc[r, 'genres_list'])
        results['diversities'].append(len(all_genres))
        
        # Coverage
        results['coverage'].update(recs)
    
    return results


# Test set
print("\nSelecting diverse test set...")
test_queries = []

# Popular anime
popular = df.nlargest(50, 'Members').index.tolist()
test_queries.extend(np.random.choice(popular, 20, replace=False))

# High-rated anime
high_rated = df[df['Score'] > 8.0].sample(min(20, len(df[df['Score'] > 8.0])), random_state=42).index.tolist()
test_queries.extend([q for q in high_rated if q not in test_queries][:10])

# Random sample
random_sample = df.sample(20, random_state=42).index.tolist()
test_queries.extend([q for q in random_sample if q not in test_queries][:10])

test_queries = list(set(test_queries))[:40]

print(f"✓ Selected {len(test_queries)} test queries")

# Run evaluation
print("\nRunning evaluation...")
results = evaluate_recommendations(test_queries, k=10)

print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)

print(f"\nQuality Metrics:")
print(f"  Average Score: {np.mean(results['avg_scores']):.3f} ± {np.std(results['avg_scores']):.3f}")
print(f"  Min: {np.min(results['avg_scores']):.3f}")
print(f"  Max: {np.max(results['avg_scores']):.3f}")

print(f"\nRelevance Metrics:")
print(f"  Avg Genre Overlap: {np.mean(results['genre_overlaps']):.2f} ± {np.std(results['genre_overlaps']):.2f}")
print(f"  Min: {np.min(results['genre_overlaps']):.2f}")
print(f"  Max: {np.max(results['genre_overlaps']):.2f}")

print(f"\nDiversity Metrics:")
print(f"  Avg Unique Genres: {np.mean(results['diversities']):.1f} ± {np.std(results['diversities']):.1f}")
print(f"  Coverage: {len(results['coverage']):,} unique anime recommended")
print(f"  Coverage Rate: {len(results['coverage']) / len(df) * 100:.2f}%")

print("\n✓ Evaluation complete!")

COMPREHENSIVE MODEL EVALUATION

Selecting diverse test set...
✓ Selected 40 test queries

Running evaluation...

EVALUATION RESULTS

Quality Metrics:
  Average Score: 7.929 ± 0.566
  Min: 6.500
  Max: 8.615

Relevance Metrics:
  Avg Genre Overlap: 1.73 ± 0.75
  Min: 0.00
  Max: 3.00

Diversity Metrics:
  Avg Unique Genres: 6.2 ± 2.6
  Coverage: 345 unique anime recommended
  Coverage Rate: 1.73%

✓ Evaluation complete!


## 3. SHAP Explainability

Use SHAP to explain individual recommendations and understand feature contributions.

In [3]:
import shap

print("SHAP EXPLAINABILITY ANALYSIS")
print("="*70)

# Select a test anime for explanation
test_idx = df[df['title'].str.contains('Death Note', case=False, na=False)].index[0]
test_title = df.loc[test_idx, 'title']

print(f"\nExplaining recommendations for: {test_title}")
print(f"Genres: {', '.join(df.loc[test_idx, 'genres_list'])}")

# Get recommendations
recs, scores = get_ltr_recommendations(test_idx, k=8)

print(f"\nTop 8 recommendations:")
print("-"*70)
for i, (rec_idx, score) in enumerate(zip(recs, scores), 1):
    rec_title = df.loc[rec_idx, 'title'][:40]
    rec_score = df.loc[rec_idx, 'Score']
    rec_genres = ', '.join(df.loc[rec_idx, 'genres_list'][:2])
    print(f"{i}. {rec_title:40s} | Score:{rec_score:.2f} | {rec_genres}")

# Extract features for SHAP
print("\n" + "="*70)
print("Computing SHAP values...")
print("="*70)

candidate_features = np.array([
    create_ranking_features(test_idx, rec_idx)
    for rec_idx in recs
])

# Feature names
feature_names = [
    'cosine_sim', 'degree', 'degree_centrality', 'pagerank', 'clustering',
    'n_studios', 'n_producers', 'studio_avg_pagerank', 'producer_avg_pagerank',
    'studio_avg_degree', 'producer_avg_degree', 'cand_score', 'cand_log_members',
    'cand_log_favorites', 'cand_popularity', 'genre_overlap', 'genre_jaccard',
    'studio_overlap', 'has_score', 'is_highly_rated'
]

# Create SHAP explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(candidate_features)

print("\n✓ SHAP values computed")

# Show top recommendation explanation
print("\n" + "="*70)
print(f"DETAILED EXPLANATION: Recommendation #1")
print("="*70)

top_rec_idx = recs[0]
top_rec_title = df.loc[top_rec_idx, 'title']
print(f"\nRecommended: {top_rec_title}")
print(f"Score: {df.loc[top_rec_idx, 'Score']:.2f}")

# Get SHAP values for top recommendation
shap_vals = shap_values[0]
features_vals = candidate_features[0]

# Create explanation dataframe
explanation_df = pd.DataFrame({
    'Feature': feature_names,
    'Value': features_vals,
    'SHAP': shap_vals
}).sort_values('SHAP', key=abs, ascending=False)

print("\nTop 10 Feature Contributions:")
print("-"*70)
for _, row in explanation_df.head(10).iterrows():
    impact = "↑" if row['SHAP'] > 0 else "↓"
    print(f"{row['Feature']:20s} | Value: {row['Value']:6.3f} | Impact: {impact} {abs(row['SHAP']):6.3f}")

# Summary statistics
print("\n" + "="*70)
print("SHAP SUMMARY STATISTICS")
print("="*70)

avg_shap = np.mean(np.abs(shap_values), axis=0)
shap_summary = pd.DataFrame({
    'Feature': feature_names,
    'Avg_Abs_SHAP': avg_shap
}).sort_values('Avg_Abs_SHAP', ascending=False)

print("\nAverage Feature Impact (across all recommendations):")
print(shap_summary.head(10).to_string(index=False))

print("\n✓ SHAP analysis complete!")

SHAP EXPLAINABILITY ANALYSIS

Explaining recommendations for: Death Note
Genres: Supernatural, Suspense

Top 8 recommendations:
----------------------------------------------------------------------
1. Shinseiki Evangelion                     | Score:8.36 | Avant Garde, Award Winning
2. Death Parade                             | Score:8.13 | Drama, Fantasy
3. Serial Experiments Lain                  | Score:8.10 | Avant Garde, Award Winning
4. Hellsing Ultimate                        | Score:8.34 | Action, Horror
5. Yuu☆Yuu☆Hakusho                          | Score:8.46 | Action, Supernatural
6. Summertime Render                        | Score:8.47 | Mystery, Supernatural
7. Psycho-Pass                              | Score:8.33 | Action, Mystery
8. Boku dake ga Inai Machi                  | Score:8.30 | Mystery, Suspense

Computing SHAP values...

✓ SHAP values computed

DETAILED EXPLANATION: Recommendation #1

Recommended: Shinseiki Evangelion
Score: 8.36

Top 10 Feature Contributions:

## 4. Human-Readable Explamnations

Generate natural language explanations for recommendations.

In [5]:
print("GENERATING HUMAN-READABLE EXPLANATIONS")
print("="*70)

def explain_recommendation(query_idx, rec_idx, shap_values, features):
    """Generate natural language explanation"""
    
    query_title = df.loc[query_idx, 'title']
    rec_title = df.loc[rec_idx, 'title']
    
    # Get top contributing features
    feature_names_list = [
        'cosine_sim', 'degree', 'degree_centrality', 'pagerank', 'clustering',
        'n_studios', 'n_producers', 'studio_avg_pagerank', 'producer_avg_pagerank',
        'studio_avg_degree', 'producer_avg_degree', 'cand_score', 'cand_log_members',
        'cand_log_favorites', 'cand_popularity', 'genre_overlap', 'genre_jaccard',
        'studio_overlap', 'has_score', 'is_highly_rated'
    ]
    
    contributions = sorted(
        zip(feature_names_list, shap_values, features),
        key=lambda x: abs(x[1]),
        reverse=True
    )
    
    # Build explanation
    explanation_parts = []
    
    # Quality
    if contributions[0][0] == 'cand_score':
        score = df.loc[rec_idx, 'Score']
        if score >= 8.5:
            explanation_parts.append(f"⭐ Exceptionally highly rated ({score:.2f}/10)")
        elif score >= 8.0:
            explanation_parts.append(f"⭐ Highly rated ({score:.2f}/10)")
        elif score >= 7.0:
            explanation_parts.append(f"✓ Well-rated ({score:.2f}/10)")
    
    # Content similarity
    for feat, shap_val, val in contributions[:10]:
        if feat == 'cosine_sim' and abs(shap_val) > 0.001:
            if val > 0.3:
                explanation_parts.append(f"📝 Very similar content and themes")
            elif val > 0.2:
                explanation_parts.append(f"📝 Similar content")
        
        elif feat == 'genre_overlap' and val > 0:
            query_genres = set(df.loc[query_idx, 'genres_list'])
            rec_genres = set(df.loc[rec_idx, 'genres_list'])
            shared = query_genres & rec_genres
            if len(shared) >= 2:
                explanation_parts.append(f"🎭 Shares genres: {', '.join(list(shared)[:2])}")
            elif len(shared) == 1:
                explanation_parts.append(f"🎭 Shares genre: {list(shared)[0]}")
        
        elif feat == 'studio_overlap' and val > 0:
            query_studios = set(df.loc[query_idx, 'studios_list'])
            rec_studios = set(df.loc[rec_idx, 'studios_list'])
            shared = query_studios & rec_studios
            if shared:
                explanation_parts.append(f"🎬 Same studio: {list(shared)[0]}")
        
        elif feat == 'is_highly_rated' and val > 0:
            if 'highly rated' not in ' '.join(explanation_parts).lower():
                explanation_parts.append(f"🏆 Community favorite")
    
    # Popularity
    for feat, shap_val, val in contributions[:10]:
        if feat == 'cand_log_members' and abs(shap_val) > 0.001:
            members = df.loc[rec_idx, 'Members']
            if members > 1000000:
                explanation_parts.append(f"👥 Very popular ({members:,.0f} members)")
    
    return explanation_parts[:4]  # Top 4 reasons


# Generate explanations for Death Note recommendations
print(f"\nRecommendation Explanations for: {test_title}")
print("="*70)

for i, (rec_idx, shap_vals, features) in enumerate(zip(recs, shap_values, candidate_features), 1):
    rec_title = df.loc[rec_idx, 'title']
    rec_score = df.loc[rec_idx, 'Score']
    
    print(f"\n{i}. {rec_title}")
    print(f"   Score: {rec_score:.2f}/10")
    
    explanations = explain_recommendation(test_idx, rec_idx, shap_vals, features)
    
    print(f"   Why recommended:")
    for exp in explanations:
        print(f"     • {exp}")

print("\n" + "="*70)
print("EXPLANATION TEMPLATE SYSTEM")
print("="*70)

print("\n✓ Natural language explanations generated")
print("\nExplanation types:")
print("  ⭐ Quality indicators (score-based)")
print("  📝 Content similarity (embedding-based)")
print("  🎭 Genre matching")
print("  🎬 Studio connections")
print("  🏆 Community validation")
print("  👥 Popularity signals")

print("\n✓ Explanations ready for user interface!")

GENERATING HUMAN-READABLE EXPLANATIONS

Recommendation Explanations for: Death Note

1. Shinseiki Evangelion
   Score: 8.36/10
   Why recommended:
     • ⭐ Highly rated (8.36/10)
     • 📝 Similar content
     • 👥 Very popular (1,971,275 members)

2. Death Parade
   Score: 8.13/10
   Why recommended:
     • ⭐ Highly rated (8.13/10)
     • 📝 Very similar content and themes
     • 🎬 Same studio: Madhouse
     • 👥 Very popular (1,884,945 members)

3. Serial Experiments Lain
   Score: 8.10/10
   Why recommended:
     • ⭐ Highly rated (8.10/10)
     • 📝 Similar content

4. Hellsing Ultimate
   Score: 8.34/10
   Why recommended:
     • ⭐ Highly rated (8.34/10)
     • 🎬 Same studio: Madhouse
     • 📝 Similar content

5. Yuu☆Yuu☆Hakusho
   Score: 8.46/10
   Why recommended:
     • ⭐ Highly rated (8.46/10)
     • 📝 Similar content

6. Summertime Render
   Score: 8.47/10
   Why recommended:
     • ⭐ Highly rated (8.47/10)
     • 📝 Similar content

7. Psycho-Pass
   Score: 8.33/10
   Why recommend

## 5. Visualizations & Summary

Create visual analysis and save all evaluation results.

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns

print("CREATING VISUALIZATIONS")
print("="*70)

# Set style
sns.set_palette("husl")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LightGBM Ranker - Evaluation Summary', fontsize=16, fontweight='bold')

# 1. Score Distribution
ax1 = axes[0, 0]
ax1.hist(results['avg_scores'], bins=20, edgecolor='black', alpha=0.7)
ax1.axvline(np.mean(results['avg_scores']), color='red', linestyle='--', 
            label=f'Mean: {np.mean(results["avg_scores"]):.2f}')
ax1.set_xlabel('Average Recommendation Score')
ax1.set_ylabel('Frequency')
ax1.set_title('Quality Distribution')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Genre Overlap Distribution
ax2 = axes[0, 1]
ax2.hist(results['genre_overlaps'], bins=15, edgecolor='black', alpha=0.7, color='orange')
ax2.axvline(np.mean(results['genre_overlaps']), color='red', linestyle='--',
            label=f'Mean: {np.mean(results["genre_overlaps"]):.2f}')
ax2.set_xlabel('Average Genre Overlap')
ax2.set_ylabel('Frequency')
ax2.set_title('Relevance Distribution')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Diversity Distribution
ax3 = axes[1, 0]
ax3.hist(results['diversities'], bins=15, edgecolor='black', alpha=0.7, color='green')
ax3.axvline(np.mean(results['diversities']), color='red', linestyle='--',
            label=f'Mean: {np.mean(results["diversities"]):.1f}')
ax3.set_xlabel('Unique Genres per Query')
ax3.set_ylabel('Frequency')
ax3.set_title('Diversity Distribution')
ax3.legend()
ax3.grid(alpha=0.3)

# 4. Top Features (SHAP)
ax4 = axes[1, 1]
top_features = shap_summary.head(8)
ax4.barh(range(len(top_features)), top_features['Avg_Abs_SHAP'], color='purple', alpha=0.7)
ax4.set_yticks(range(len(top_features)))
ax4.set_yticklabels(top_features['Feature'])
ax4.set_xlabel('Average |SHAP Value|')
ax4.set_title('Top 8 Feature Importance (SHAP)')
ax4.grid(alpha=0.3, axis='x')

plt.tight_layout()
viz_path = DATA_DIR / 'evaluation_summary.png'
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Visualization saved: {viz_path}")

plt.close()

# Save evaluation results
print("\n" + "="*70)
print("SAVING EVALUATION RESULTS")
print("="*70)

evaluation_results = {
    'model': 'LightGBM LambdaRank',
    'test_size': len(test_queries),
    'metrics': {
        'quality': {
            'avg_score': float(np.mean(results['avg_scores'])),
            'std_score': float(np.std(results['avg_scores'])),
            'min_score': float(np.min(results['avg_scores'])),
            'max_score': float(np.max(results['avg_scores']))
        },
        'relevance': {
            'avg_genre_overlap': float(np.mean(results['genre_overlaps'])),
            'std_genre_overlap': float(np.std(results['genre_overlaps'])),
            'min_overlap': float(np.min(results['genre_overlaps'])),
            'max_overlap': float(np.max(results['genre_overlaps']))
        },
        'diversity': {
            'avg_unique_genres': float(np.mean(results['diversities'])),
            'std_unique_genres': float(np.std(results['diversities'])),
            'coverage': len(results['coverage']),
            'coverage_rate': float(len(results['coverage']) / len(df))
        }
    },
    'shap_analysis': {
        'top_features': shap_summary.head(10).to_dict('records')
    },
    'explanation_types': [
        'Quality indicators (score-based)',
        'Content similarity (embedding-based)',
        'Genre matching',
        'Studio connections',
        'Community validation',
        'Popularity signals'
    ]
}

import json
with open(PROCESSED_DIR / 'evaluation_results.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print("✓ Evaluation results saved: evaluation_results.json")

print("\n" + "="*70)
print("NOTEBOOK 09 COMPLETE")
print("="*70)

print("\nDeliverables:")
print("  ✓ evaluation_results.json - Complete metrics")
print("  ✓ evaluation_summary.png - Visual analysis")
print("  ✓ SHAP explanations - Feature importance")
print("  ✓ Explanation templates - User-facing reasons")

print("\nKey Findings:")
print(f"  ✓ Avg Quality: {np.mean(results['avg_scores']):.2f}/10")
print(f"  ✓ Avg Relevance: {np.mean(results['genre_overlaps']):.2f} genre overlap")
print(f"  ✓ Diversity: {np.mean(results['diversities']):.1f} unique genres")
print(f"  ✓ Coverage: {len(results['coverage'])} anime ({len(results['coverage'])/len(df)*100:.2f}%)")

print("\nTop 3 Most Important Features:")
for i, row in shap_summary.head(3).iterrows():
    print(f"  {i+1}. {row['Feature']}: {row['Avg_Abs_SHAP']:.4f}")

CREATING VISUALIZATIONS

✓ Visualization saved: data\evaluation_summary.png

SAVING EVALUATION RESULTS
✓ Evaluation results saved: evaluation_results.json

NOTEBOOK 09 COMPLETE

Deliverables:
  ✓ evaluation_results.json - Complete metrics
  ✓ evaluation_summary.png - Visual analysis
  ✓ SHAP explanations - Feature importance
  ✓ Explanation templates - User-facing reasons

Key Findings:
  ✓ Avg Quality: 7.93/10
  ✓ Avg Relevance: 1.73 genre overlap
  ✓ Diversity: 6.2 unique genres
  ✓ Coverage: 345 anime (1.73%)

Top 3 Most Important Features:
  12. cand_score: 0.2626
  20. is_highly_rated: 0.0899
  14. cand_log_favorites: 0.0170
